# 02 - Silver: limpando com Dynamic Tables

**Objetivo:** criar tabelas **limpas e padronizadas** que se atualizam sozinhas quando a Bronze muda.

**Analogia:** a Dynamic Table é como uma **célula de planilha com fórmula**. Você escreve a regra uma vez (`=LIMPAR(A1)`) e, quando o valor de origem muda, o resultado se recalcula. Aqui a "fórmula" é um `SELECT`.

```mermaid
flowchart LR
  B[BRONZE.CLIENTES] -->|"remove duplicados, padroniza"| S[SILVER.CLIENTES]
  B2[BRONZE.PEDIDOS] -->|"converte datas e valores"| S2[SILVER.PEDIDOS]
```

**Tempo estimado:** 25 minutos

In [ ]:
%%sql -r dataframe_1
-- Contexto da sessao: quem sou eu, qual motor e qual gaveta vou usar.
-- Rode esta celula SEMPRE que abrir o notebook.
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE ESPM_WH;
USE DATABASE ESPM_HOL;
USE SCHEMA SILVER;

## Os 4 problemas que vamos resolver

| Problema na Bronze | Solução na Silver |
|---|---|
| 15 clientes duplicados | `QUALIFY ROW_NUMBER()`: fica só a versão mais recente |
| Datas em 3 formatos | `TRY_TO_DATE` tenta cada formato |
| E-mail com maiúscula e espaço | `LOWER(TRIM(email))` |
| Valores como texto | `TRY_TO_NUMBER(..., 12, 2)`: número com 2 casas decimais, exato para dinheiro |

**TARGET_LAG** é o "atraso máximo aceitável". `DOWNSTREAM` quer dizer: atualize quando alguém depois de mim (a Gold) precisar.

### EXERCICIO 1 - Complete a Dynamic Table de clientes: padronize o e-mail e defina o TARGET_LAG

Complete os `???` na célula abaixo.

**Dica:** a função que deixa tudo minúsculo é `LOWER`. Para o TARGET_LAG, use o valor explicado logo acima (atualiza quando a Gold precisar).

**Travou?** Abra o Cortex Code e peça:
> Complete esta Dynamic Table: o e-mail deve ficar em minúsculas e o TARGET_LAG deve ser DOWNSTREAM

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE DYNAMIC TABLE SILVER.CLIENTES
  TARGET_LAG = ???
  WAREHOUSE = ESPM_WH
  COMMENT = 'Clientes unicos e padronizados'
AS
SELECT
    cliente_id,
    INITCAP(TRIM(nome))                         AS nome,
    ???(TRIM(email))                            AS email,        -- padroniza o e-mail
    cidade,
    uf,
    COALESCE(TRY_TO_DATE(LEFT(data_cadastro, 10), 'YYYY-MM-DD'), TRY_TO_DATE(data_cadastro, 'DD/MM/YYYY')) AS data_cadastro,
    canal_aquisicao_id,
    TRY_TO_TIMESTAMP(atualizado_em)             AS atualizado_em
FROM BRONZE.CLIENTES
-- Para cada cliente_id, fica so a linha mais recente
QUALIFY ROW_NUMBER() OVER (PARTITION BY cliente_id ORDER BY TRY_TO_TIMESTAMP(atualizado_em) DESC) = 1;

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE DYNAMIC TABLE SILVER.PEDIDOS
  TARGET_LAG = DOWNSTREAM
  WAREHOUSE = ESPM_WH
  COMMENT = 'Pedidos com tipos corretos'
AS
SELECT
    pedido_id,
    cliente_id,
    loja_id,
    canal_venda,
    COALESCE(TRY_TO_DATE(LEFT(data_pedido, 10), 'YYYY-MM-DD'), TRY_TO_DATE(data_pedido, 'DD/MM/YYYY')) AS data_pedido,
    status,
    NULLIF(cupom, '')                          AS cupom,
    TRY_TO_NUMBER(valor_frete, 12, 2)          AS valor_frete,
    TRY_TO_NUMBER(valor_total, 12, 2)          AS valor_total,   -- NULL quando veio vazio
    -- Regra de negocio: so conta como venda se nao foi cancelado/devolvido e tem valor
    (status NOT IN ('CANCELADO', 'DEVOLVIDO') AND TRY_TO_NUMBER(valor_total, 12, 2) IS NOT NULL) AS pedido_valido
FROM BRONZE.PEDIDOS;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE DYNAMIC TABLE SILVER.ITENS_PEDIDO
  TARGET_LAG = DOWNSTREAM WAREHOUSE = ESPM_WH
AS
SELECT item_id, pedido_id, produto_id,
       TRY_TO_NUMBER(quantidade)                                   AS quantidade,
       TRY_TO_NUMBER(preco_unitario, 12, 2)                        AS preco_unitario,
       TRY_TO_NUMBER(desconto, 12, 2)                              AS desconto,
       TRY_TO_NUMBER(quantidade) * TRY_TO_NUMBER(preco_unitario, 12, 2)
         - COALESCE(TRY_TO_NUMBER(desconto, 12, 2), 0)             AS valor_liquido
FROM BRONZE.ITENS_PEDIDO;

CREATE OR REPLACE DYNAMIC TABLE SILVER.PRODUTOS
  TARGET_LAG = DOWNSTREAM WAREHOUSE = ESPM_WH
AS
SELECT produto_id, nome_produto, categoria, TRY_TO_NUMBER(preco_lista, 12, 2) AS preco_lista
FROM BRONZE.PRODUTOS;

CREATE OR REPLACE DYNAMIC TABLE SILVER.CANAIS_CAMPANHA
  TARGET_LAG = DOWNSTREAM WAREHOUSE = ESPM_WH
AS
SELECT canal_id, nome_canal, tipo_canal, TRY_TO_NUMBER(custo_mensal, 12, 2) AS custo_mensal
FROM BRONZE.CANAIS_CAMPANHA;

CREATE OR REPLACE DYNAMIC TABLE SILVER.CAMPANHAS_DISPAROS
  TARGET_LAG = DOWNSTREAM WAREHOUSE = ESPM_WH
AS
SELECT disparo_id, canal_id, cliente_id,
       COALESCE(TRY_TO_DATE(LEFT(data_disparo, 10), 'YYYY-MM-DD'), TRY_TO_DATE(data_disparo, 'DD/MM/YYYY')) AS data_disparo,
       abriu  = 'S'                          AS abriu,      -- vira verdadeiro/falso
       clicou = 'S'                          AS clicou,
       NULLIF(pedido_convertido_id, '')      AS pedido_convertido_id
FROM BRONZE.CAMPANHAS_DISPAROS;

### Checkpoint: a limpeza funcionou?

In [ ]:
%%sql -r dataframe_5
-- Esperado: 285 | 0 | 0 | 300 | 10
SELECT
  (SELECT COUNT(*) FROM SILVER.CLIENTES)                                  AS clientes_unicos,
  (SELECT COUNT_IF(email <> LOWER(TRIM(email))) FROM SILVER.CLIENTES)     AS emails_sujos,
  (SELECT COUNT_IF(data_pedido IS NULL) FROM SILVER.PEDIDOS)              AS datas_invalidas,
  (SELECT COUNT(*) FROM SILVER.PEDIDOS)                                   AS pedidos,
  (SELECT COUNT_IF(valor_total IS NULL) FROM SILVER.PEDIDOS)              AS pedidos_sem_valor;

In [ ]:
%%sql -r dataframe_6
-- Antes x depois: as mesmas datas, cruas e convertidas
SELECT b.pedido_id, b.data_pedido AS data_bronze, s.data_pedido AS data_silver
FROM BRONZE.PEDIDOS b JOIN SILVER.PEDIDOS s USING (pedido_id)
LIMIT 6;

In [ ]:
%%sql -r dataframe_7
-- Estado das Dynamic Tables
SHOW DYNAMIC TABLES IN SCHEMA ESPM_HOL.SILVER;

**O que observar:** em `SHOW DYNAMIC TABLES`, a coluna `refresh_mode` mostra se o Snowflake atualiza de forma **INCREMENTAL** (só o que mudou) ou **FULL** (tudo de novo).

Para ver o grafo: menu **Monitoring > Dynamic Tables** > clique em `CLIENTES` > aba **Graph**.

**Próximo:** `03_gold_streams_tasks` - transformar dado limpo em indicador de negócio.